# Discrete Distributions
**Topic:** Random Variables & Distributions

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the conditions under which you should use a Binomial versus a Poisson distribution
- **Calculate** probabilities using Binomial and Poisson PMFs
- **Explain** the relationship between these two distributions as the Poisson limit of the Binomial

> **Tip:** Start by moving the **Binomial n slider** to very large values while keeping p small, watch how the Binomial PMF gradually takes on the shape of the Poisson distribution.

---
## How we got here

In *09: Discrete Random Variables* we built the general framework: a PMF assigns probabilities to each possible count outcome. That framework had no constraints on *which* PMF was correct. This notebook introduces two named distributions, Binomial and Poisson, each representing a specific type of counting process that appears throughout data science.

---
## Why this matters for data science

Binomial and Poisson distributions are the workhorses of count modeling. A Binomial is the right model any time you count "successes" across a fixed number of independent trials, click-through rates, defect counts, A/B test outcomes. A Poisson models event arrivals when the rate is known but the exact count is uncertain, requests per second to an API, messages per hour in a chat app, customer arrivals per minute at a register.

---
## Try it yourself

In [ ]:
from plotly.subplots import make_subplots
from tkh_utils import make_output, make_dropdown, make_slider, make_int_slider

# ── Scenario text per distribution ────────────────────────────────────────────
_CONTEXTS = {
    "Binomial": {
        "simple": (
            "Flip a coin <b>n</b> times. Each flip independently lands heads with "
            "probability <b>p</b>. The total number of heads is Binomial."
        ),
        "applied": (
            "Same model, bigger stage: send a campaign email to <b>n</b> subscribers — "
            "each person independently clicks with probability <b>p</b>. "
            "Count the clicks."
        ),
        "rule": "Use Binomial when: fixed number of trials, two outcomes per trial, trials are independent.",
    },
    "Poisson": {
        "simple": (
            "Buses arrive at your stop at an average rate of <b>λ</b> per hour. "
            "The number of buses that show up in any given hour is Poisson."
        ),
        "applied": (
            "Same model, different setting: a customer support chat receives messages "
            "at an average rate of <b>λ</b> per hour. Count the messages."
        ),
        "rule": "Use Poisson when: events arrive at a steady average rate, no fixed upper bound on the count.",
    },
}

_FORMULAS = {
    "Binomial": "P(X = k) = C(n,k) · p<sup>k</sup> · (1−p)<sup>n−k</sup>",
    "Poisson":  "P(X = k) = e<sup>−λ</sup> · λ<sup>k</sup> / k!",
}

# ── Controls ──────────────────────────────────────────────────────────────────
out       = make_output()
dist_dd   = make_dropdown(["Binomial", "Poisson"], "Binomial", "Distribution:")
n_sl      = make_int_slider(1, 100, 1, 20,  "Trials (n):")
p_sl      = make_slider(0.01, 0.99, 0.01, 0.30, "Prob (p):")
lam_sl    = make_slider(0.5, 20.0, 0.5, 4.0, "Rate (λ):")

context_box = widgets.HTML()

_binom_ctrls = VBox([n_sl, p_sl])
_pois_ctrls  = VBox([lam_sl])
_pois_ctrls.layout.display = "none"

# ── Render ────────────────────────────────────────────────────────────────────
def render(change=None):
    dist = dist_dd.value
    rng  = np.random.default_rng(42)

    if dist == "Binomial":
        n, p    = n_sl.value, p_sl.value
        mu, var = n * p, n * p * (1 - p)
        k_lo    = max(0, int(stats.binom.ppf(0.001, n, p)) - 1)
        k_hi    = min(n, int(stats.binom.ppf(0.999, n, p)) + 2)
        k_vals  = np.arange(k_lo, k_hi + 1)
        pmf     = stats.binom.pmf(k_vals, n, p)
        draws   = rng.binomial(n, p, 50)
        title   = f"Binomial(n={n}, p={p:.2f})"
        params  = f"n = {n},  p = {p:.2f}"
    else:
        lam     = lam_sl.value
        mu = var = lam
        k_lo    = 0
        k_hi    = int(stats.poisson.ppf(0.999, lam)) + 2
        k_vals  = np.arange(k_lo, k_hi + 1)
        pmf     = stats.poisson.pmf(k_vals, lam)
        draws   = rng.poisson(lam, 50)
        title   = f"Poisson(λ={lam:.1f})"
        params  = f"λ = {lam:.1f}"

    dtick  = max(1, (k_hi - k_lo) // 16)
    jitter = rng.uniform(-0.22, 0.22, len(draws))
    ctx    = _CONTEXTS[dist]

    context_box.value = (
        f'<div style="font-size:13px; line-height:1.8; padding:12px 16px; '
        f'background:#f8f9fa; border-left:4px solid {PALETTE["primary"]}; '
        f'border-radius:4px; margin-bottom:8px;">'
        f'<b style="font-size:14px;">{dist} distribution</b><br>'
        f'<b>Simple example:</b> {ctx["simple"]}<br>'
        f'<b>Applied example:</b> {ctx["applied"]}<br>'
        f'<span style="color:#777; font-size:12px;">{ctx["rule"]}</span><br><br>'
        f'<b>PMF:</b> {_FORMULAS[dist]}&emsp;|&emsp;'
        f'<b>Parameters:</b> {params}&emsp;|&emsp;'
        f'E[X] = {mu:.2f},&ensp;Var(X) = {var:.2f}'
        f'</div>'
    )

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.06,
    )

    fig.add_trace(go.Bar(
        x=k_vals, y=pmf,
        marker_color=PALETTE["primary"], opacity=0.85,
        showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=draws, y=jitter,
        mode="markers",
        marker=dict(
            color=PALETTE["secondary"], size=9, opacity=0.70,
            line=dict(color="white", width=1),
        ),
        showlegend=False,
    ), row=2, col=1)

    layout = base_layout(
        title=f"{title}  —  E[X] = {mu:.2f},  Var(X) = {var:.2f}",
        xaxis_title="",
        yaxis_title="P(X = k)",
    )
    layout.update(
        height=480,
        xaxis=dict(tickmode="linear", dtick=dtick, gridcolor="#eee", zeroline=False),
        xaxis2=dict(
            title="k  (number of occurrences)",
            tickmode="linear", dtick=dtick, gridcolor="#eee", zeroline=False,
        ),
        yaxis2=dict(visible=False, range=[-0.5, 0.5]),
    )
    fig.update_layout(layout)

    fig.add_annotation(
        text="50 random draws from this distribution",
        xref="x2 domain", yref="y2 domain",
        x=0.5, y=0.98, xanchor="center", yanchor="top",
        showarrow=False,
        font=dict(size=11, family=FONT["family"], color="#888"),
    )

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

# ── Observers ─────────────────────────────────────────────────────────────────
def _on_dist_change(change):
    _binom_ctrls.layout.display = "" if dist_dd.value == "Binomial" else "none"
    _pois_ctrls.layout.display  = "none" if dist_dd.value == "Binomial" else ""
    render()

dist_dd.observe(_on_dist_change, names="value")
n_sl.observe(render,   names="value")
p_sl.observe(render,   names="value")
lam_sl.observe(render, names="value")

# ── Display ───────────────────────────────────────────────────────────────────
controls = VBox([dist_dd, _binom_ctrls, _pois_ctrls])
display(VBox([controls, context_box, out]))
render()


---
## What's happening?

Both distributions count occurrences, but they model slightly different situations.

| Distribution | Parameters | When to use |
|-------------|-----------|-------------|
| Binomial(n, p) | n = trials, p = success prob | Fixed number of independent trials, each can succeed or fail |
| Poisson(λ) | λ = average rate | Counts over a continuous interval, no fixed number of trials |

$$P(X=k \mid n,p) = \binom{n}{k} p^k (1-p)^{n-k} \qquad E[X]=np, \; \text{Var}(X)=np(1-p)$$

$$P(X=k \mid \lambda) = \frac{e^{-\lambda}\lambda^k}{k!} \qquad E[X]=\lambda, \; \text{Var}(X)=\lambda$$

### The Poisson limit

As n → ∞ and p → 0 with np = λ held constant, Binomial(n, p) converges to Poisson(λ). This is why the Poisson is used when n is huge (all possible website visitors) and p is tiny (probability any one visitor arrives in a given second).

Verify this in the widget: set n = 500, p = 0.008, and compare the Binomial PMF to Poisson(λ = 4), they should be nearly identical.

---
## Real-world example: Website click-through rates

An online ad is shown to 200 users per hour. Each user independently clicks with probability 0.03 (a 3% click-through rate, typical for display advertising). The number of clicks per hour follows a Binomial(200, 0.03) distribution.

The chart overlays the Binomial PMF and its Poisson approximation (λ = 6) to show how close they are in practice. Notice:

- **Notice:** The two distributions are nearly indistinguishable, in practice, the Poisson is used because it requires only one parameter (λ) instead of two (n, p)
- **Notice:** The most probable number of clicks is 5 or 6, with the distribution roughly symmetric around E[X] = 6
- **Notice:** There is real probability of getting 0 clicks in an hour, and also real probability of getting 12 or more, the tails matter for capacity planning

> **Discussion question:** If the ad platform charges per impression and you are paid per click, which matters more for your business model, E[X] or the tail probabilities P(X ≤ 2)? How would you use this distribution to set a minimum guarantee?

In [3]:
# ── Binomial vs Poisson for ad click-through ─────────────────────────────────
n_users = 200
p_click = 0.03
lam_approx = n_users * p_click  # = 6.0

k_vals = np.arange(0, 20)
binom_pmf  = stats.binom.pmf(k_vals, n_users, p_click)
poisson_pmf = stats.poisson.pmf(k_vals, lam_approx)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=k_vals - 0.2, y=binom_pmf, width=0.35,
    marker_color=PALETTE["primary"], name=f"Binomial(n={n_users}, p={p_click})",
))
fig.add_trace(go.Bar(
    x=k_vals + 0.2, y=poisson_pmf, width=0.35,
    marker_color=PALETTE["secondary"], opacity=0.8,
    name=f"Poisson(λ={lam_approx:.0f})",
))
layout = base_layout(
    title=f"Ad Clicks per Hour — Binomial vs Poisson Approximation  (E[X] = {lam_approx})",
    xaxis_title="Number of Clicks (k)",
    yaxis_title="P(X = k)",
)
layout.update(barmode="overlay", xaxis=dict(tickmode="linear", dtick=1))
fig.update_layout(layout)
fig.show()

### When to use which discrete distribution

| Situation | Distribution | Key check |
|-----------|-------------|----------|
| Fixed n trials, count successes | Binomial(n, p) | Are trials independent? Is p constant? |
| Count events in a time/area interval | Poisson(λ) | Is rate λ constant? Are events independent? |
| How many trials until first success? | Geometric(p) | Same independence assumption as Binomial |
| Count of rare defects in a large batch | Poisson(λ) | n huge, p tiny: Poisson is the limit |
| Survey: exactly k of n chose option A | Binomial(n, p) | Sampling with replacement assumed |

---
## Key takeaway

> **Binomial counts successes across a fixed number of trials; Poisson counts events over a continuous interval — and when trials are plentiful but individual success is rare, they give the same answer.**

---
*Next up: Continuous Distributions — when outcomes are not counts but measurements that can take any value on the number line*